# **Purpose**

**Task:** context + answer → question

This notebook fine-tunes `microsoft/Phi-3.5-mini-instruct` using `LoRA/QLoRA` on a subset of SQuAD to generate a question given a context passage and an answer span.

**Runtime:** Runtime → Change runtime type → GPU. Phi-3.5-mini-instruct is ~3.8B params, so in 4-bit (QLoRA) it comfortably fits a T4 (16GB) with the settings below; use a larger GPU if you have one available or want a larger effective batch size.

## **Install dependencies**

In [1]:
!pip install -qU \
 transformers==5.16.1 \
 accelerate==1.14.0 \
 peft==0.18.1 \
 bitsandbytes==0.48.2 \
 datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.2 MB/s eta 0:00:00


## **Imports**

In [2]:
import numpy as np
import torch
import transformers
import accelerate
import peft

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

from kaggle_secrets import UserSecretsClient
import os

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)

torch: 2.10.0+cu128
transformers: 5.16.1
accelerate: 1.14.0
peft: 0.18.1


## **Load the API Keys and Tokens**

In [3]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")   # must match the exact secret name you set in Kaggle Secrets

os.environ["HF_TOKEN"] = hf_token

## **Config**

Tweak these for experimenting. `TRAIN_SUBSET_SIZE` / `VAL_SUBSET_SIZE` control how much of SQuAD you use - start small to sanity-check the pipeline before scaling up.

`USE_4BIT` toggles QLoRA (4-bit base model + LoRA adapters) vs. plain LoRA (16-bit base model + LoRA adapters). Leave it `True` unless you have a GPU with plenty of headroom.

**Phi-3.5 specifics:**
- Phi-3.5-mini-instruct fuses attention projections into `qkv_proj`/`o_proj` and the MLP into `gate_up_proj`/`down_proj` (rather than Qwen's separate `q_proj`/`k_proj`/`v_proj`/`gate_proj`/`up_proj`), so `LORA_TARGET_MODULES` below is different from the Qwen notebook.
- Since Phi-3.5-mini is ~2.5x larger than Qwen2.5-1.5B, `per_device_train_batch_size` is reduced and `gradient_accumulation_steps` increased to keep the effective batch size the same (16) while staying inside T4 memory.

In [4]:
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"   # ~3.8B params. Swap to "microsoft/Phi-3-mini-4k-instruct" for the earlier
                                                    # generation, or a larger Phi model if you have more GPU room
MAX_INPUT_LENGTH = 512                      # context+answer prompt length (same budget as the Flan-T5/Qwen notebooks)
MAX_TARGET_LENGTH = 96                      # questions are short
MAX_SEQ_LENGTH = MAX_INPUT_LENGTH + MAX_TARGET_LENGTH + 32   # causal LM sees prompt+completion in one sequence
TRAIN_SUBSET_SIZE = 10000                   # subset of SQuAD train split, set to None for full data
VAL_SUBSET_SIZE = 10
TRAIN_EPOCH_SIZE = 1
OUTPUT_DIR = "/content/phi3.5-mini-qg-lora"
SEED = 42

# --- PEFT / QLoRA config ---
USE_4BIT = True             # QLoRA (4-bit base model) if True, plain LoRA (16-bit base model) if False
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "qkv_proj", "o_proj",       # fused attention projections in Phi-3/Phi-3.5
    "gate_up_proj", "down_proj",  # fused MLP projections in Phi-3/Phi-3.5
]

SYSTEM_PROMPT = (
    "You are a question generation assistant. Given a context passage and a target answer, "
    "generate a single question whose correct answer is exactly the target answer."
)

# *****IMPORTNT*****
# Change the Huggingface pushing directory below

## **Load SQuAD and take a subset**

Uses the Hugging Face `squad` dataset. Swap to `"squad_v2"` if you also want unanswerable examples (note: `squad_v2` has empty answer lists for some examples, which the preprocessing below already handles gracefully).

In [5]:
raw = load_dataset("squad")

train_ds = raw["train"].shuffle(seed=SEED)
val_ds = raw["validation"].shuffle(seed=SEED)

if TRAIN_SUBSET_SIZE:
    train_ds = train_ds.select(range(TRAIN_SUBSET_SIZE))
if VAL_SUBSET_SIZE:
    val_ds = val_ds.select(range(VAL_SUBSET_SIZE))

print(train_ds)
print(val_ds)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10000
})
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10
})


In [6]:
# Peek at one example
train_ds[0]

{'id': '573173d8497a881900248f0c',
 'title': 'Egypt',
 'context': 'The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.',
 'question': 'What percentage of Egyptians polled support death penalty for those leaving Islam?',
 'answers': {'text': ['84%'], 'answer_start': [468]}}

## **Load tokenizer and (quantized) model**

When `USE_4BIT` is on, the base model is loaded in 4-bit NF4 precision via `bitsandbytes` and prepared for k-bit training - this is the "QLoRA" part. LoRA adapters (added in the next section) are trained on top in full/half precision regardless.

Phi-3.5-mini's chat template and architecture are natively supported by recent `transformers` versions, so no `trust_remote_code` flag is needed here (add `trust_remote_code=True` to both `from_pretrained` calls if you're on an older `transformers` and hit a "not registered" style error).

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # right padding for training with causal LMs

compute_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

model.config.use_cache = False   # required alongside gradient checkpointing

config.json: 0.00B [00:00, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

## **Configure LoRA**

Wraps the base model with LoRA adapters on the (fused) attention and MLP projection layers. Only these adapter weights (a small fraction of total parameters) get trained and saved - the base model stays frozen.

In [8]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 25,165,824 || all params: 3,846,245,376 || trainable%: 0.6543


## **Preprocessing**

Builds the same "Target Answer / Context" prompt as the Qwen/Flan-T5 notebooks, but wraps it in Phi-3.5's chat template (system + user turn) and appends the gold question as the assistant turn. Since this is a causal LM, prompt and completion are packed into a single sequence - the loss is masked (`-100`) over the prompt tokens so the model is only trained to predict the question itself.

**Tip:** for the "answer highlighting" trick used in a lot of QG literature, wrap the answer span inside the context with a marker (e.g. `<hl> {answer} <hl>`) before building the prompt - this can measurably improve which part of the context the model attends to.

In [9]:
def build_user_message(context, answer_text):
    return (
        f"Target Answer: {answer_text}\n"
        f"Generate a question from the following context where the target answer is the correct answer. "
        f"Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\n"
        f"Context: {context}"
    )


def preprocess(examples):
    all_input_ids, all_labels, all_attention_mask = [], [], []

    for context, answers, question in zip(examples["context"], examples["answers"], examples["question"]):
        answer_text = answers["text"][0] if len(answers["text"]) > 0 else ""
        user_msg = build_user_message(context, answer_text)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        full_text = prompt_text + question + tokenizer.eos_token

        prompt_ids = tokenizer(
            prompt_text, add_special_tokens=False, truncation=True, max_length=MAX_INPUT_LENGTH
        )["input_ids"]
        full_ids = tokenizer(
            full_text, add_special_tokens=False, truncation=True, max_length=MAX_SEQ_LENGTH
        )["input_ids"]

        labels = list(full_ids)
        prompt_len = min(len(prompt_ids), len(full_ids))
        for i in range(prompt_len):
            labels[i] = -100

        all_input_ids.append(full_ids)
        all_labels.append(labels)
        all_attention_mask.append([1] * len(full_ids))

    return {
        "input_ids": all_input_ids,
        "labels": all_labels,
        "attention_mask": all_attention_mask,
    }


tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)

# Dynamically pads input_ids/attention_mask/labels per batch (labels padded with -100).
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

## **Training arguments**

On Colab, set `bf16=True` if your GPU supports it (T4 only supports `fp16`; A100/L4/newer support `bf16`) - adjust the precision flags below if needed. Adjust batch size / gradient accumulation if you hit out-of-memory errors. `paged_adamw_8bit` keeps the optimizer state memory-efficient, which matters most when `USE_4BIT=True`.

Batch size is smaller than the Qwen notebook (2 vs. 4) with accumulation doubled (8 vs. 4) to hold the same effective batch size of 16 while accounting for Phi-3.5-mini's larger footprint.

In [10]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    weight_decay=0.01,
    num_train_epochs=TRAIN_EPOCH_SIZE,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=50,
    save_total_limit=2,
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to="none",   # set to "wandb"/"tensorboard" if you use experiment tracking
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    processing_class=tokenizer,
    data_collator=data_collator,
)

## **Train the model**

In [11]:
trainer.train()

Step,Training Loss
50,1.308154
100,1.126062
150,1.132937
200,1.083763
250,1.106106
300,1.120853
350,1.099603
400,1.045841
450,1.053491
500,1.046448


TrainOutput(global_step=625, training_loss=1.108671810913086, metrics={'train_runtime': 9639.5805, 'train_samples_per_second': 1.037, 'train_steps_per_second': 0.065, 'total_flos': 7.474983375727411e+16, 'train_loss': 1.108671810913086, 'epoch': 1.0})

## **Save the LoRA adapter**

Because this is a PEFT model, `save_model` / `save_pretrained` write out only the (small) LoRA adapter weights and config, not the full base model - the base model can always be re-downloaded from the Hub and combined with this adapter.

In [12]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to {OUTPUT_DIR}")

Saved LoRA adapter to /content/phi3.5-mini-qg-lora


## **Push the LoRA adapter to Huggingface Hub**

In [13]:
model.push_to_hub("gaurav-dey/phi3.5-mini-qg-lora")
tokenizer.push_to_hub("gaurav-dey/phi3.5-mini-qg-lora")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/gaurav-dey/phi3.5-mini-qg-lora/commit/b4c0d87bd2eeb6f4055858d449f26511acb32f04', commit_message='Upload tokenizer', commit_description='', oid='b4c0d87bd2eeb6f4055858d449f26511acb32f04', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gaurav-dey/phi3.5-mini-qg-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='gaurav-dey/phi3.5-mini-qg-lora'), pr_revision=None, pr_num=None)

## **Merge the model**

In [14]:
merged_model = model.merge_and_unload()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


## **Push the merged model to Huggingface Hub**

In [15]:
merged_model.push_to_hub("gaurav-dey/phi3.5-mini-qg")
tokenizer.push_to_hub("gaurav-dey/phi3.5-mini-qg")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/gaurav-dey/phi3.5-mini-qg/commit/a801aabdf3468b90b48a05d2ae118f09867857a6', commit_message='Upload tokenizer', commit_description='', oid='a801aabdf3468b90b48a05d2ae118f09867857a6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gaurav-dey/phi3.5-mini-qg', endpoint='https://huggingface.co', repo_type='model', repo_id='gaurav-dey/phi3.5-mini-qg'), pr_revision=None, pr_num=None)

Optional: mount Google Drive and copy the checkpoint there so it persists after the Colab runtime disconnects.

## **Sanity-check generations**

In [16]:
model.eval()
tokenizer.padding_side = "left"   # left padding is what you want for batched generation

sample = val_ds.select(range(5))
for ex in sample:
    answer_text = ex["answers"]["text"][0] if ex["answers"]["text"] else ""
    user_msg = build_user_message(ex["context"], answer_text)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            do_sample=False,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    generated_question = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    print(f"Answer:              {answer_text}")
    print(f"Gold question:       {ex['question']}")
    print(f"Generated question:  {generated_question}")
    print("-" * 80)

Answer:              1852
Gold question:       In what year did Massachusetts first require children to be educated in schools?
Generated question:  In what year was compulsory education first established in Massachusetts, setting the stage for debates on private schooling in the United States?
--------------------------------------------------------------------------------
Answer:              1962
Gold question:       When were stromules discovered?
Generated question:  In what year were stromules, a functional feature of plant cell plastids, first observed and initially dismissed by some plant biologists as artifactual?
--------------------------------------------------------------------------------
Answer:              Horace Walpole
Gold question:       Which artist who had a major influence on the Gothic Revival is represented in the V&A's British galleries?
Generated question:  Who was a major influence on the Gothic Revival and had works of art from their collection displayed i

## **Next steps**

- Set `USE_4BIT = False` to compare plain LoRA (16-bit base model) against QLoRA on quality and training speed.
- Add the answer-highlighting (`<hl>` marker) preprocessing variant and compare against this baseline.
- Add a round-trip QA-consistency evaluation metric for a semantic-quality signal beyond ROUGE/BLEU.